In [ ]:
from uq_physicell.model_analysis import ModelAnalysisContext, run_simulations

db_path = "ObsData.db"  # Path to the database file
model_config = {"ini_path": "uq_config.ini", "struc_name": "ModelA"} # Example model configuration
sampler = 'Custom'  # Example sampler
params_info = { # Example parameters information
    'cell_cycle_entry': {'ref_value': 1440.0, 'perturbation': None},
    'apoptosis_rate': {'ref_value': 5.787e-05, 'perturbation': None}
}  
qoi_funcs = {
    'live_cell_count': lambda df: len( df[ df['dead'] == False ]),
    'sum_dead_cell_count': lambda mcds_ts: sum(map(lambda mcds: len(mcds.get_cell_df()[mcds.get_cell_df()['dead'] == True]), mcds_ts))    
}
# Create Model Analysis Context
context = ModelAnalysisContext(db_path, model_config, sampler, params_info, qoi_funcs, num_workers=8)
# Sample parameter sets and run simulations
context.dic_samples = {0: {'cell_cycle_entry': 1440.0, 'apoptosis_rate': 5.787e-05}}
run_simulations(context)

Inserting parameter: cell_cycle_entry with properties: {'ref_value': 1440.0, 'perturbation': None, 'lower_bound': None, 'upper_bound': None}
Inserting parameter: apoptosis_rate with properties: {'ref_value': 5.787e-05, 'perturbation': None, 'lower_bound': None, 'upper_bound': None}
Inserting {'live_cell_count': "lambda df: len( df[ df['dead'] == False ])", 'cum_dead_cell_count': "lambda mcds_ts: sum(map(lambda mcds: len(mcds.get_cell_df()[mcds.get_cell_df()['dead'] == True]), mcds_ts))"} QoIs into the database
Computed QoIs at time 0.0: {'live_cell_count': 511}
Computed QoIs at time 240.0: {'live_cell_count': 582}
Computed QoIs at time 480.0: {'live_cell_count': 624}
Computed QoIs at time 720.0: {'live_cell_count': 651}
Computed QoIs at time 960.0: {'live_cell_count': 695}
Computed QoIs at time 1200.0: {'live_cell_count': 727}
Computed QoIs at time 1440.0: {'live_cell_count': 770}
Computed QoIs at time 1680.0: {'live_cell_count': 811}
Computed QoIs at time 1920.0: {'live_cell_count': 8

In [ ]:
import pandas as pd
from uq_physicell.database.ma_db import load_output
# Load the simulation results and extract cell counts over time
df_qois_data = load_output(db_path)
df_all_data = pd.DataFrame()
for index, row in df_qois_data.iterrows():
    df_temp = row['Data']
    df_all_data = pd.concat([df_all_data, df_temp], ignore_index=True)

# Aggregate with multiple summary statistics
df_aggregated = df_all_data.groupby('time').agg({
    'live_cell_count': ['mean', 'std', 'min', 'max', 'count'],
    'sum_dead_cell_count': ['mean', 'std', 'min', 'max', 'count']
}).reset_index()

# Flatten column names for easier access
df_aggregated.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                          for col in df_aggregated.columns.values]

time = df_aggregated['time'].values
live = df_aggregated['live_cell_count_mean'].values
dead = df_aggregated['sum_dead_cell_count_mean'].values
df_aggregated.rename(columns={'time': 'Time', 'live_cell_count_mean': 'Live_Cells', 'sum_dead_cell_count_mean': 'Sum_Dead_Cells'}, inplace=True)
display(df_aggregated)
df_aggregated.to_csv("ObsData.csv", index=False)

,Time,Live_Cells,live_cell_count_std,live_cell_count_min,live_cell_count_max,live_cell_count_count,Cum_Dead_Cells,cum_dead_cell_count_std,cum_dead_cell_count_min,cum_dead_cell_count_max,cum_dead_cell_count_count
0,0.0,511.0,0.000000,511,511,5,NaN,NaN,NaN,NaN,0
1,240.0,581.4,6.767570,571,590,5,NaN,NaN,NaN,NaN,0
2,480.0,621.4,4.335897,614,624,5,NaN,NaN,NaN,NaN,0
3,720.0,651.8,7.854935,642,664,5,NaN,NaN,NaN,NaN,0
4,960.0,693.4,8.080842,680,702,5,NaN,NaN,NaN,NaN,0
5,1200.0,725.0,9.591663,709,735,5,NaN,NaN,NaN,NaN,0
6,1440.0,767.6,9.099451,752,776,5,NaN,NaN,NaN,NaN,0
7,1680.0,803.0,14.764823,777,811,5,NaN,NaN,NaN,NaN,0
8,1920.0,847.6,14.081903,823,859,5,NaN,NaN,NaN,NaN,0
9,2160.0,890.0,10.606602,875,905,5,NaN,NaN,NaN,NaN,0
